# Global PIQA preprocessing notebook

This notebook prepares the HuggingFace dataset `mrlbenchmarks/global-piqa-nonparallel` into the local JSONL format used by the multilingual MCQ loader.

It does four things:

1. Lists all available dataset configs.
2. Identifies all configs matching the five target languages: Chinese, Hindi, Italian, Spanish, Russian.
3. Loads all relevant configs per language, concatenates them, shuffles with seed `42`, and saves one JSONL file per language.
4. Prints per-language counts and a few example prompts for quality checks.

In [30]:
from pathlib import Path
import json
import random
from collections import defaultdict, Counter
import json
from datasets import get_dataset_config_names, load_dataset, concatenate_datasets
import re
from tqdm.auto import tqdm

In [37]:
DATASET_NAME = "mrlbenchmarks/global-piqa-nonparallel"
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
OUTPUT_DIR = REPO_ROOT / "data/processed/global_piqa"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)
SEED = 42
N_EXAMPLES_TO_PRINT = 3
LABEL_MAP = {0: "A", 1: "B", "0": "A", "1": "B"}

Repo root: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual
Output dir: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_piqa


## 1. List all available configs

to find the datasets for all languages/countries using these languages

In [6]:
all_configs = get_dataset_config_names(DATASET_NAME)
print(f"Total configs: {len(all_configs)}")

for cfg in all_configs:
    print(cfg)

Total configs: 136
acm_arab
acq_arab
aeb_arab
afb_arab
als_latn
amh_ethi
apc_arab_jord
apc_arab_leba
apc_arab_pale
apc_arab_syri
arb_arab
arq_arab
ars_arab
ary_arab
arz_arab
asm_beng
azj_latn
bam_latn
bcc_arab
bel_cyrl
ben_beng
ben_latn
bgc_deva
bho_deva
bos_latn
bra_deva
bsk_arab
btx_latn
bul_cyrl
cat_latn
ces_latn
ckb_arab
ckm_latn
cls_deva
cmn_hans
cmn_hant
dan_latn
deu_latn
dhd_deva
dje_latn
ekk_latn
ekp_latn
ell_grek
eng_latn
eus_latn
fao_latn
fin_latn
fra_latn_cana
fra_latn_fran
glg_latn
guj_gujr
hau_latn
haw_latn
heb_hebr
hin_deva
hin_latn
hrv_latn
hun_latn
hye_armn
ibo_latn
idu_latn
ind_latn
isl_latn
iso_latn
ita_latn
jav_latn
jpn_jpan
kan_knda
kan_latn
kat_geor
kaz_cyrl
kin_latn
kir_cyrl
kor_hang
lin_latn
lit_latn
luo_latn
mal_mlym
mar_deva
mkd_cyrl
mni_beng
mni_mtei
nag_latn
nld_latn
nno_latn
nob_latn
npi_deva
ory_orya
pan_guru
pcm_latn
pes_arab
plt_latn
pol_latn
por_latn_braz
por_latn_port
ron_latn
rus_cyrl
rwr_deva
sin_latn
sin_sinh
slk_latn
slk_latn_sari
slv_latn
slv_latn_

In [7]:
GLOBAL_PIQA_CONFIGS = {
    "chinese": [
        "cmn_hans",   # Mandarin Chinese, Simplified
        "cmn_hant",   # Mandarin Chinese, Traditional
    ],
    "hindi": [
        "hin_deva",   # Hindi, Devanagari
        "hin_latn",   # Hindi, Latin transliteration
    ],
    "italian": [
        "ita_latn",   # Italian
    ],
    "spanish": [
        "spa_latn_mexi",  # Mexican Spanish
        "spa_latn_peru",  # Peruvian Spanish
        "spa_latn_spai",  # Spain Spanish
    ],
    "russian": [
        "rus_cyrl",   # Russian
    ],
}

## 2. Process

In [32]:
BAD_TEXT_VALUES = {"", "nan", "none", "null", "n/a", "na", "?", "-", "undefined"}
SEED = 42
TIMESTAMP_RE = re.compile(r"\b\d{4}-\d{2}-\d{2}\s+00:00:00\b")
DATE_RE = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
ARTIFACT_RE = re.compile(r"(Unnamed:\s*\d+|\[object Object\])", re.IGNORECASE)

In [33]:
def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip()).lower()


def text_is_missing_or_bad(x):
    if x is None:
        return True
    t = str(x).strip()
    low = t.lower()

    if low in BAD_TEXT_VALUES:
        return True
    if ARTIFACT_RE.search(t):
        return True
    if TIMESTAMP_RE.search(t):
        return True

    return False


def label_is_valid(label):
    try:
        return int(label) in {0, 1}
    except Exception:
        return False


def row_is_malformed(row):
    required = ["prompt", "solution0", "solution1", "label"]

    for key in required:
        if key not in row:
            return True, f"missing_field:{key}"

    if text_is_missing_or_bad(row["prompt"]):
        return True, "bad_prompt"

    if text_is_missing_or_bad(row["solution0"]):
        return True, "bad_solution0"

    if text_is_missing_or_bad(row["solution1"]):
        return True, "bad_solution1"

    if not label_is_valid(row["label"]):
        return True, "invalid_label"

    if normalize_text(row["solution0"]) == normalize_text(row["solution1"]):
        return True, "duplicate_options"

    # Optional but useful: remove very short/degenerate text.
    if len(str(row["prompt"]).strip()) < 5:
        return True, "prompt_too_short"

    if len(str(row["solution0"]).strip()) < 2:
        return True, "solution0_too_short"

    if len(str(row["solution1"]).strip()) < 2:
        return True, "solution1_too_short"

    return False, None


def row_key(row):
    return (
        normalize_text(row["prompt"]),
        normalize_text(row["solution0"]),
        normalize_text(row["solution1"]),
    )

def clean_dataset_rows(rows):
    """
    rows: list of dict-like examples after loading and adding config_name/source_split.
    Returns:
        cleaned_rows, cleaning_stats
    """
    stats = Counter()

    # 1. Remove malformed rows.
    valid_rows = []
    for row in rows:
        bad, reason = row_is_malformed(row)
        if bad:
            stats[f"removed_{reason}"] += 1
            continue
        valid_rows.append(row)

    # 2. Remove duplicate example_id if present.
    seen_example_ids = set()
    dedup_by_id = []

    for row in valid_rows:
        ex_id = row.get("example_id")

        if ex_id is not None:
            ex_id = str(ex_id).strip()
            if ex_id in seen_example_ids:
                stats["removed_duplicate_example_id"] += 1
                continue
            seen_example_ids.add(ex_id)

        dedup_by_id.append(row)

    # 3. Remove conflicting duplicates by normalized prompt/options.
    labels_by_key = defaultdict(set)
    for row in dedup_by_id:
        labels_by_key[row_key(row)].add(int(row["label"]))

    conflicting_keys = {k for k, labels in labels_by_key.items() if len(labels) > 1}

    no_conflicts = []
    for row in dedup_by_id:
        if row_key(row) in conflicting_keys:
            stats["removed_conflicting_duplicate"] += 1
            continue
        no_conflicts.append(row)

    # 4. Remove exact duplicate prompt/options/label.
    seen_full = set()
    cleaned = []

    for row in no_conflicts:
        full_key = row_key(row) + (int(row["label"]),)
        if full_key in seen_full:
            stats["removed_duplicate_prompt_options_label"] += 1
            continue

        seen_full.add(full_key)
        cleaned.append(row)

    stats["kept"] = len(cleaned)
    stats["input"] = len(rows)
    stats["removed_total"] = len(rows) - len(cleaned)

    return cleaned, stats


def label_to_letter(label):
    if int(label) == 0:
        return "A"
    if int(label) == 1:
        return "B"
    raise ValueError(f"Unexpected label: {label}")


def make_prompt(row):
    return (
        f"{row['prompt']}\n\n"
        f"A) {row['solution0']}\n"
        f"B) {row['solution1']}"
    )


def convert_example(row, idx, language, config_name):
    metadata = {
        "config": config_name,
    }

    for key in ["approx_cultural_score", "example_id", "source_split"]:
        if key in row and row[key] is not None:
            metadata[key] = row[key]

    return {
        "id": f"global_piqa_{language}_{idx}",
        "dataset": "global_piqa",
        "language": language,
        "task_type": "mcq",
        "prompt": make_prompt(row),
        "answer": label_to_letter(row["label"]),
        "metadata": metadata,
    }

def load_all_splits_for_config(config_name):
    ds_dict = load_dataset(DATASET_NAME, config_name)

    print(f"{config_name}: splits = {list(ds_dict.keys())}")

    splits = []
    for split_name, split_ds in ds_dict.items():
        split_ds = split_ds.add_column("source_split", [split_name] * len(split_ds))
        splits.append(split_ds)

    return concatenate_datasets(splits)

In [34]:
all_language_counts = {}
all_cleaning_stats = {}

for language, configs in GLOBAL_PIQA_CONFIGS.items():
    print("=" * 80)
    print(f"Processing language: {language}")
    print(f"Configs: {configs}")

    datasets_for_language = []

    for cfg in configs:
        ds = load_all_splits_for_config(cfg)
        ds = ds.add_column("config_name", [cfg] * len(ds))
        datasets_for_language.append(ds)

    combined = concatenate_datasets(datasets_for_language)
    combined = combined.shuffle(seed=SEED)

    rows = [dict(row) for row in combined]
    cleaned_rows, stats = clean_dataset_rows(rows)

    output_path = OUTPUT_DIR / f"global_piqa_{language}_all.jsonl"
    with output_path.open("w", encoding="utf-8") as f:
        for idx, row in enumerate(cleaned_rows):
            ex = convert_example(
                row=row,
                idx=idx,
                language=language,
                config_name=row["config_name"],
            )
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    all_language_counts[language] = len(cleaned_rows)
    all_cleaning_stats[language] = dict(stats)
    print(f"Input rows: {stats['input']}")
    print(f"Removed total: {stats['removed_total']}")
    print(f"Kept: {stats['kept']}")
    print("Removal reasons:")
    for key, value in sorted(stats.items()):
        if key.startswith("removed_") and key != "removed_total":
            print(f"  {key}: {value}")

    print(f"Saved {len(cleaned_rows)} examples to {output_path}\n\n")

Processing language: chinese
Configs: ['cmn_hans', 'cmn_hant']
cmn_hans: splits = ['test']
cmn_hant: splits = ['test']
Input rows: 200
Removed total: 0
Kept: 200
Removal reasons:
Saved 200 examples to /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/global_piqa/global_piqa_chinese_all.jsonl


Processing language: hindi
Configs: ['hin_deva', 'hin_latn']
hin_deva: splits = ['test']
hin_latn: splits = ['test']
Input rows: 200
Removed total: 0
Kept: 200
Removal reasons:
Saved 200 examples to /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/global_piqa/global_piqa_hindi_all.jsonl


Processing language: italian
Configs: ['ita_latn']
ita_latn: splits = ['test']
Input rows: 100
Removed total: 0
Kept: 100
Removal reasons:
Saved 100 examples to /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/global_piqa/global_piqa_italian_all.jsonl


Proces

In [35]:
print("Final counts:")
for language, count in all_language_counts.items():
    print(f"{language}: {count}")

Final counts:
chinese: 200
hindi: 200
italian: 100
spanish: 300
russian: 100


In [36]:
# Print a few examples per language for quality inspection
N_EXAMPLES = 3
for language in GLOBAL_PIQA_CONFIGS:
    path = OUTPUT_DIR / f"global_piqa_{language}_all.jsonl"

    print("=" * 80)
    print(f"Examples for {language}: {path}")

    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= N_EXAMPLES:
                break
            ex = json.loads(line)
            print("-" * 80)
            print(f"id: {ex['id']}")
            print(f"answer: {ex['answer']}")
            print(f"metadata: {ex['metadata']}")
            print(ex["prompt"])

Examples for chinese: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/global_piqa/global_piqa_chinese_all.jsonl
--------------------------------------------------------------------------------
id: global_piqa_chinese_0
answer: B
metadata: {'config': 'cmn_hant', 'approx_cultural_score': 1, 'example_id': 'group0008_ex000057_cmn_hant_0_v1', 'source_split': 'test'}
老馬拿著柳琴向編鐘砸去，柳琴會怎樣？

A) 編鐘會被砸爛。
B) 柳琴會被砸爛。
--------------------------------------------------------------------------------
id: global_piqa_chinese_1
answer: B
metadata: {'config': 'cmn_hans', 'approx_cultural_score': 1, 'example_id': 'group0048_ex000096_cmn_hans_2_v1', 'source_split': 'test'}
油泼辣子的做法是

A) 把油泼在加热后的食材上。
B) 把加热后的油泼在食材上。
--------------------------------------------------------------------------------
id: global_piqa_chinese_2
answer: A
metadata: {'config': 'cmn_hant', 'approx_cultural_score': 1, 'example_id': 'group0008_ex000028_cmn_hant_0_v1', 'source_split': 'te